# 🚢 Teleport Intelligence Dashboard
## Cargo Shortfall Prediction & Revenue Optimisation

**Project Owner:** Alyaa (Luuna Sari)  
**Started:** 2026  
**Goal:** Build a Python-powered analytics dashboard that tracks cargo commitment performance, predicts shortfall risk, and surfaces revenue opportunities.

---

### 🗺️ What We're Building (Your Roadmap)

| Module | What You'll Add |
|--------|----------------|
| 1–2 | Load data, write your first functions ✅ (today!) |
| 3–4 | Loop through routes, summarise performance |
| 5–6 | Flag contracts as On Track / At Risk / Shortfall |
| 7 | Interactive route lookup |
| 8–9 | Charts — revenue leakage, upgrade demand |
| 10 | Save reports to files |
| 11 | Write tests for your functions |
| 12–13 | Time series — forecast next 4 weeks |
| 14–15 | ML — predict shortfall risk score |
| Capstone | Streamlit dashboard — portfolio ready |

---

### 📌 Today's Goal (Module 1–2)
1. Upload and load the dataset
2. Look at what's inside
3. Write your first function: `calculate_shortfall()`
4. Write your second function: `flag_status()`
5. Run both on real data and see results

> **ADHD tip:** Do one cell at a time. Run it. Look at the output. Then move to the next. You don't have to finish today.

---
## Step 1: Upload Your Dataset

Run the cell below. It will show an **Upload** button.  
Upload the file `teleport_cargo_data.csv` that came with this notebook.

In [ ]:
# Upload the CSV file to Colab
from google.colab import files

uploaded = files.upload()  # Click 'Choose Files' and upload teleport_cargo_data.csv
print("✅ File uploaded successfully!")

---
## Step 2: Load the Data

We use **pandas** — a Python library that lets you work with tables of data (like Excel, but in code).  
`pd.read_csv()` reads a CSV file and turns it into a **DataFrame** — a table you can slice, filter, and analyse.

In [ ]:
import pandas as pd  # pd is the standard short name for pandas

# Load the dataset into a DataFrame called 'df'
df = pd.read_csv("teleport_cargo_data.csv")

print(f"✅ Loaded {len(df)} records")
print(f"📅 Date range: {df['week_start'].min()} to {df['week_start'].max()}")
print(f"🛣️  Routes: {df['route'].nunique()} unique routes")
print(f"🏢 Customers: {df['customer'].nunique()} unique customers")

---
## Step 3: Peek at the Data

`.head()` shows the first 5 rows — like opening a spreadsheet and seeing the top rows.  
Always do this first. It tells you what columns you have and what the data looks like.

In [ ]:
# Show the first 5 rows
df.head()

In [ ]:
# Show all column names and their data types
# This tells you: what do I have to work with?
df.info()

In [ ]:
# Show basic statistics for all number columns
# min, max, mean, std — a quick health check on your data
df.describe().round(2)

---
## Step 4: Your First Function — `calculate_shortfall()`

This is the core of the whole project.  
A **function** is a reusable block of code that takes inputs and returns an output.  

```
def function_name(input1, input2):
    # do something
    return result
```

Our first function: given agreed tonnage and booked tonnage, calculate the shortfall fee.

In [ ]:
def calculate_shortfall(agreed_kg, booked_kg, rate_per_kg, penalty_rate=0.20):
    """
    Calculate the shortfall fee for a cargo contract.

    Parameters:
    - agreed_kg     : tonnage committed in the contract (kg)
    - booked_kg     : tonnage actually booked (kg)
    - rate_per_kg   : base rate charged per kg (RM)
    - penalty_rate  : shortfall penalty as % of base rate (default 20%)

    Returns:
    - shortfall_fee : fee charged for the shortfall (RM)
    """
    shortfall_kg = max(0, agreed_kg - booked_kg)  # can't be negative
    shortfall_fee = shortfall_kg * rate_per_kg * penalty_rate
    return round(shortfall_fee, 2)


# --- Test it manually first ---
# Example: agreed 1000kg, only booked 750kg, rate RM5/kg
fee = calculate_shortfall(agreed_kg=1000, booked_kg=750, rate_per_kg=5.0)
print(f"Shortfall fee: RM {fee}")
# Expected: (1000-750) * 5.0 * 0.20 = RM 250.00

# Edge case: what if they booked MORE than agreed?
fee2 = calculate_shortfall(agreed_kg=1000, booked_kg=1200, rate_per_kg=5.0)
print(f"Overbooked fee: RM {fee2}")  # Should be 0 — no penalty for overdelivery

---
## Step 5: Your Second Function — `flag_status()`

This uses **if/elif/else** to categorise each contract.  
You already know this logic from your shortfall analysis work — now it's Python.

In [ ]:
def flag_status(agreed_kg, booked_kg):
    """
    Flag a contract's performance status.

    Returns:
    - 'On Track'  : booked >= 100% of agreed
    - 'At Risk'   : booked 85–99% of agreed
    - 'Shortfall' : booked < 85% of agreed
    """
    utilisation = booked_kg / agreed_kg  # ratio: 0.0 to 1.0+

    if utilisation >= 1.0:
        return "On Track"
    elif utilisation >= 0.85:
        return "At Risk"
    else:
        return "Shortfall"


# --- Test it ---
print(flag_status(1000, 1000))  # On Track
print(flag_status(1000, 900))   # At Risk  (90%)
print(flag_status(1000, 700))   # Shortfall (70%)

---
## Step 6: Apply Both Functions to Your Real Data

Now we apply these functions across all 1,296 rows using pandas.  
`.apply()` runs a function on every row of a column — like dragging a formula down in Excel, but faster.

In [ ]:
# Apply calculate_shortfall() to every row
df["my_shortfall_fee"] = df.apply(
    lambda row: calculate_shortfall(
        row["agreed_tonnage_kg"],
        row["booked_tonnage_kg"],
        row["base_rate_per_kg"]
    ),
    axis=1  # axis=1 means apply row by row
)

# Apply flag_status() to every row
df["my_status"] = df.apply(
    lambda row: flag_status(
        row["agreed_tonnage_kg"],
        row["booked_tonnage_kg"]
    ),
    axis=1
)

print("✅ Functions applied to all rows!")
df[["customer", "route", "agreed_tonnage_kg", "booked_tonnage_kg",
    "my_shortfall_fee", "my_status"]].head(10)

---
## Step 7: First Business Insight

Let's answer a real question:  
**Which route has the highest total shortfall fees?**

This is `.groupby()` — the pandas equivalent of a pivot table in Excel.

In [ ]:
# Group by route, sum shortfall fees, sort descending
route_shortfall = (
    df.groupby("route")["my_shortfall_fee"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

route_shortfall.columns = ["Route", "Total Shortfall Fee (RM)"]
route_shortfall["Total Shortfall Fee (RM)"] = route_shortfall["Total Shortfall Fee (RM)"].apply(
    lambda x: f"RM {x:,.2f}"
)

print("📊 Shortfall Fees by Route (6 months):")
print(route_shortfall.to_string(index=False))

In [ ]:
# Contract status breakdown across all records
status_summary = df["my_status"].value_counts()
total = len(df)

print("📋 Contract Status Summary:")
for status, count in status_summary.items():
    pct = count / total * 100
    bar = "█" * int(pct / 2)
    print(f"  {status:<12} {count:>4} contracts  ({pct:.1f}%)  {bar}")

print(f"\n💰 Total shortfall fees at risk: RM {df['my_shortfall_fee'].sum():,.2f}")

---
## Step 8: Your First Chart 🎉

One bar chart. Route vs shortfall fee. This is the start of your dashboard.

In [ ]:
import matplotlib.pyplot as plt

# Get numeric version of route shortfall
route_nums = (
    df.groupby("route")["my_shortfall_fee"]
    .sum()
    .sort_values(ascending=True)  # ascending so highest is at top
)

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.barh(route_nums.index, route_nums.values, color="#E53E3E")
ax.set_xlabel("Total Shortfall Fee (RM)", fontsize=12)
ax.set_title("🚨 Shortfall Fees by Route — 6 Month View", fontsize=14, fontweight="bold")

# Add value labels on bars
for bar, val in zip(bars, route_nums.values):
    ax.text(val + 500, bar.get_y() + bar.get_height()/2,
            f"RM {val:,.0f}", va="center", fontsize=10)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig("shortfall_by_route.png", dpi=150)
plt.show()
print("✅ Chart saved as shortfall_by_route.png")

---
## ✅ You're Done for Today!

Here's what you just built:

- ✅ Loaded a real dataset with 1,296 cargo records
- ✅ Wrote `calculate_shortfall()` — your first Python function
- ✅ Wrote `flag_status()` — using if/elif/else logic
- ✅ Applied both functions across all rows using pandas
- ✅ Generated a business insight: which route bleeds the most in shortfall fees
- ✅ Made your first chart

---

### 🔜 Next Session (Module 3–4)

You'll add:
- A loop that prints a performance report for each route
- A list comprehension that filters only 'Shortfall' contracts
- Summary stats per customer: who shortfalls the most?

---

> **Remember:** You don't have to do it all at once.
> One cell. Run it. Look at it. Next cell.
> That's how the whole dashboard gets built. 🚢